In [ ]:
import random
import numpy as np
import os

def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except ImportError:
        pass

set_seed(42)


In [ ]:
#pip install sentence-transformers pandas numpy scikit-learn torch

# Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import re
import torch
import pickle
from sentence_transformers import SentenceTransformer, util

# Load Dataset

In [ ]:
df = pd.read_csv("input/amazon-eco-friendly-products-dataset/amazon_eco-friendly_products.csv")

df.head()

# Preprocessing and Cleaning

In [ ]:
# Check Null Values
df.isnull().sum()

In [ ]:
# Filling null values with empty string
for col in df.columns:
    df[col] = df[col].fillna('')

df.isnull().sum()

In [ ]:
# Cheack duplicate values 
df.duplicated().sum()

In [ ]:
df.info()

In [ ]:
df.head(2).T

In [ ]:
# Defining function to clean text columns
def clean_text(text):
    text = str(text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'[^\w\s&\'%-]', '', text)
    text = re.sub(r'([!?.])\1+', r'\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = text.lower()
    return text

In [ ]:
text_columns = ["title", "name", "category", "material", "brand", "description"]

# Combine columns into a single text field
df["combined_text"] = df[[col for col in text_columns]].agg(" ".join, axis=1)

# Apply cleaning to combined_text
df["combined_text"] = df["combined_text"].apply(clean_text)
df.head(2).T

# Load a pretrained model

Here I have used **Sentence Transformer** **(BERT)** for **semantic search** in this e-commerce search engine project. By embedding user queries and product descriptions into semantic vectors, we achieve more accurate and context-aware search results. This approach effectively handles synonyms, variations, and multilingual inputs, significantly enhancing the user experience and driving better search outcomes.

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Model is running on: {model.device}")


# Generate Product Embeddings

In [ ]:
product_embeddings = model.encode(df["combined_text"], convert_to_tensor = True)
product_embeddings[0]

# Save Embeddings

In [ ]:
# Save as .pkl
os.makedirs("output", exist_ok=True)
with open("output/sentence_transformer_model.pkl", "wb") as f:
    pickle.dump(model, f)
print("Model saved as sentence_transformer_model.pkl")

# Load it back for testing
with open("output/sentence_transformer_model.pkl", "rb") as f:
    model = pickle.load(f)
model = model.to(device) 
print(f"Loaded model is running on: {model.device}")

# Search Engine

In [ ]:
from sentence_transformers.util import cos_sim

# Define a function for search product
def search_products(query, model, product_embeddings, df, top_k=5):
    # Ensure the dataset and embeddings are not empty
    if len(df) == 0 or len(product_embeddings) == 0:
        raise ValueError("Dataset or product embeddings are empty. Cannot perform search.")

    # Encode the query (returns a tensor on GPU)
    query_embedding = model.encode([clean_text(query)], convert_to_tensor=True)[0]

    # Convert query_embedding to NumPy array (move to CPU)
    query_embedding = query_embedding.cpu().numpy()

    # Ensure product_embeddings is a NumPy array
    if isinstance(product_embeddings, torch.Tensor):
        product_embeddings = product_embeddings.cpu().numpy()

    # Compute cosine similarity (both inputs are NumPy arrays)
    similarities = util.cos_sim(query_embedding, product_embeddings).flatten()

    # Debugging: Check the type and content of similarities
    print("Type of similarities:", type(similarities))
    print("Shape of similarities:", similarities.shape)
    print("Sample similarities:", similarities[:5])

    # Check for invalid similarities
    if len(similarities) == 0:
        raise ValueError("Cosine similarities are empty. Check dataset or query.")

    # Convert similarities to NumPy array if it’s not already
    if isinstance(similarities, torch.Tensor):
        similarities = similarities.cpu().numpy()

    # Check for NaN values
    if np.any(np.isnan(similarities)):
        raise ValueError("Cosine similarities contain NaN values. Check for zero embeddings in the dataset.")

    # Get top-k indices
    top_k_indices = np.argsort(similarities)[-top_k:][::-1]

    # Return top-k products and scores
    return df.iloc[top_k_indices][["title", "brand", "description"]], similarities[top_k_indices]

# Test
query = "board games" # Straws, cutlary, toothbrush
try:
    top_products, scores = search_products(query, model, product_embeddings, df)

    # Create DataFrame 
    recommendations_df = pd.DataFrame(top_products)
    recommendations_df['Score'] = scores

    print("Query:", query)
    print("Top Recommendations:\n", recommendations_df)
except ValueError as e:
    print(f"Error: {e}")